# Router experiments on the v3 dataset — train, validate, compare with v2

The same harness as `route_experiments.ipynb`, pointed at the composed v3
artifact (`dataset_v3.parquet`, 45/45/10, certified tiers) instead of the v2
labels. Edit the **config** cell, Run All. New here: §8 trains v2 and v3
routers under identical config and cross-evaluates them on each other's
held-out rows, and §9 scores the served router on the frozen eval reserve —
rows no selection ever touched.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from hybrid_search_rrf_dataset.router import (
    Representation,
    RouterExperiment,
    StrategyRouter,
)

DATA = Path("data")
V3_LABELS = DATA / "v3" / "dataset_v3.parquet"
V3_CATALOG = DATA / "v3" / "catalog_v3.parquet"
V2_LABELS = DATA / "route_labels" / "labels.parquet"

# ---- config: edit and Run All ----
REPRESENTATION = Representation.ENGINEERED
ENCODER = None
PROTOCOL = 'random_within_lane'   # or 'holdout_lane'
ALL_ROWS = 'decisive'   # 'decisive' = clear winners | 'recommended' | 'all'
MAX_CLASS_SHARE = None

exp = RouterExperiment(labels_path=V3_LABELS, catalog_path=V3_CATALOG,
                       encoder=ENCODER)

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1 — train on v3
from hybrid_search_rrf_dataset.router import _derive_engineered, _margin, _routes_differ

train, test = exp.split(PROTOCOL)
router = StrategyRouter(REPRESENTATION, encoder=ENCODER).fit(
    train, all_rows=ALL_ROWS, max_class_share=MAX_CLASS_SHARE
)
print('thresholds (dense, sparse):', tuple(round(t, 3) for t in router.thresholds))

mode = {False: 'decisive', True: 'recommended'}.get(ALL_ROWS, ALL_ROWS)
fit_rows = (
    train if mode == 'all'
    else train[_routes_differ(train)] if mode == 'recommended'
    else train[_margin(train) >= router.decisive_margin]
)
eng = _derive_engineered(fit_rows)
coef = router.coefficients().set_index('feature')
coef['fires'] = [(eng[f] > 0).sum() if f in eng.columns else None for f in coef.index]
coef.round(3).nlargest(8, 'sparse_weight')

thresholds (dense, sparse): (0.5, 0.5)


/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

,dense_weight,sparse_weight,fires
feature,,,
query_corpus.avg_idf,-0.971,1.055,1235
query_corpus.vocab_overlap,-0.456,0.444,1235
unknown_token_rate.unknown_token_rate,-0.281,0.320,1445
morphology.word_variation_share,-0.334,0.298,2362
query_corpus.max_idf,-0.173,0.235,1235
subword_fragmentation.max_pieces_per_word,-0.134,0.165,8031
derived.min_zipf,-0.111,0.119,6962
structured_identifiers.stock_ticker_like,-0.136,0.114,307


In [4]:
# 2 — validate: the six-column headroom table on v3
cols = [
    'protocol', 'representation', 'all_rows', 'max_class_share', 'n_test_decisive',
    'const_dense_only', 'const_pure_rrf', 'const_sparse_only',
    'oracle', 'router', 'headroom_captured', 't_dense', 't_sparse',
]
result = exp.run(
    representations=[REPRESENTATION],
    all_rows=ALL_ROWS,
    max_class_share=MAX_CLASS_SHARE,
)
result[cols].round(3)

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, fitting]/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly f

,protocol,representation,all_rows,max_class_share,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured,t_dense,t_sparse
0,random_within_lane,engineered,decisive,None,1961,0.504,0.244,0.500,0.992,0.586,0.169,0.55,0.3
1,holdout_lane,engineered,decisive,None,463,0.436,0.268,0.538,1.000,0.490,-0.103,0.55,0.4


In [5]:
# 3 — use: route your own queries with the section-1 router
my_queries = [
    'Who likes Curling?',
    'what are the side effects of DHA',
    'CVE-2021-44228 log4j remote code execution',
    'http://localhost.com',
]
for q in my_queries:
    e = router.explain(q)
    print(f"{e['route']!s:12s} p_dense={e['p_dense']:.2f} p_sparse={e['p_sparse']:.2f}  {q}")

sparse_only  p_dense=0.32 p_sparse=0.63  Who likes Curling?
pure_rrf     p_dense=0.47 p_sparse=0.50  what are the side effects of DHA
sparse_only  p_dense=0.51 p_sparse=0.54  CVE-2021-44228 log4j remote code execution
sparse_only  p_dense=0.47 p_sparse=0.52  http://localhost.com


In [6]:
# 4 — serve: refit on ALL v3 rows (not comparable to section 2's held-out numbers)
data = exp.load()
served = StrategyRouter(REPRESENTATION, encoder=ENCODER, delta=0.05).fit(
    data, all_rows='all', max_class_share=MAX_CLASS_SHARE
)
served.tune_thresholds(data)
print('serving thresholds (dense, sparse):', tuple(round(t, 3) for t in served.thresholds))
for q in my_queries:
    e = served.explain(q)
    print(f"{e['route']!s:12s} p_dense={e['p_dense']:.2f} p_sparse={e['p_sparse']:.2f}  {q}")

/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

serving thresholds (dense, sparse): (0.5, 0.1)
sparse_only  p_dense=0.45 p_sparse=0.60  Who likes Curling?
dense_only   p_dense=0.57 p_sparse=0.42  what are the side effects of DHA
sparse_only  p_dense=0.43 p_sparse=0.54  CVE-2021-44228 log4j remote code execution
sparse_only  p_dense=0.32 p_sparse=0.63  http://localhost.com


In [7]:
# 5 — margin hedge on v3 (delta tuned on train, judged held-out)
import numpy as np

from hybrid_search_rrf_dataset.router import _mean_objective, _route_from_probs

tune_frame = train[_routes_differ(train)]
p_d, p_s = router._probabilities(tune_frame)
deltas = np.round(np.arange(0.0, 0.32, 0.002), 2)[::-1]
tune_scores = [
    _mean_objective(tune_frame, _route_from_probs(p_d, p_s, 0.5, 0.5, delta=d))
    for d in deltas
]
best_delta = float(deltas[int(np.argmax(tune_scores))])
print(f'best delta on train: {best_delta:.2f}')

decisive_test = test[_margin(test) >= router.decisive_margin]
pt_d, pt_s = router._probabilities(decisive_test)
for label, (td, ts, d) in {
    'tuned thresholds, delta=0': (*router.thresholds, 0.0),
    'symmetric (0.5, 0.5), delta=0': (0.5, 0.5, 0.0),
    f'symmetric + delta={best_delta:.2f}': (0.5, 0.5, best_delta),
}.items():
    routes = _route_from_probs(pt_d, pt_s, td, ts, delta=d)
    score = _mean_objective(decisive_test, routes)
    n_rrf = sum(r.value == 'pure_rrf' for r in routes)
    print(f'{label:32s} held-out: {score:.3f}  (rrf fired on {n_rrf}/{len(routes)})')

/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

best delta on train: 0.00
tuned thresholds, delta=0        held-out: 0.571  (rrf fired on 88/1961)
symmetric (0.5, 0.5), delta=0    held-out: 0.571  (rrf fired on 88/1961)
symmetric + delta=0.00           held-out: 0.571  (rrf fired on 88/1961)


/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

# 6 — Acceptability heads on v3 (SPEC d60 form)

Same as the v2 notebook: three binary heads on `ok_*`, cheapest acceptable
route serves. On v3 the tied mass is smaller by construction, so watch
whether the cost advantage over argmax narrows.

In [8]:
import numpy as np
import pandas as pd

from hybrid_search_rrf_dataset.fusion import SERVING_COST, StrategyName
from hybrid_search_rrf_dataset.labels import AcceptabilityLabels
from hybrid_search_rrf_dataset.router import AcceptabilityRouter, _mean_objective, _margin

acc = AcceptabilityRouter(REPRESENTATION, encoder=ENCODER).fit(train)
view = AcceptabilityLabels(test).frame()
answerable = view[view["serve"].notna()].reset_index(drop=True)
print(f"test: {len(answerable):,} answerable of {len(view):,}")

acc_routes = acc.predict_routes(answerable)
argmax_routes = router.predict_routes(answerable)
decisive_mask = (_margin(answerable) >= router.decisive_margin).to_numpy()
serve_oracle = [StrategyName(s) for s in answerable["serve"]]

def readout(policy, routes):
    routes = list(routes)
    mix = pd.Series([r.value for r in routes]).value_counts(normalize=True)
    return {
        "policy": policy,
        "objective (answerable)": _mean_objective(answerable, routes),
        "objective (decisive)": _mean_objective(
            answerable[decisive_mask], [r for r, m in zip(routes, decisive_mask) if m]),
        "serve agreement": float(np.mean([r == s for r, s in zip(routes, serve_oracle)])),
        "mean cost": float(np.mean([SERVING_COST[r] for r in routes])),
        **{f"% {s.value}": float(mix.get(s.value, 0.0)) for s in StrategyName},
    }

pd.DataFrame([
    readout("serve oracle (view)", serve_oracle),
    readout("acceptability heads", acc_routes),
    readout("argmax router (section 1)", argmax_routes),
    *(readout(f"const {s.value}", [s] * len(answerable)) for s in StrategyName),
]).round(3)

test: 7,192 answerable of 7,343


,policy,objective (answerable),objective (decisive),serve agreement,mean cost,% dense_only,% pure_rrf,% sparse_only
0,serve oracle (view),0.560,0.992,1.000,0.292,0.250,0.021,0.729
1,acceptability heads,0.379,0.544,0.431,0.755,0.543,0.106,0.351
2,argmax router (section 1),0.377,0.571,0.479,0.606,0.505,0.051,0.444
3,const dense_only,0.350,0.504,0.250,1.000,1.000,0.000,0.000
4,const pure_rrf,0.352,0.244,0.021,2.000,0.000,1.000,0.000
5,const sparse_only,0.320,0.500,0.729,0.000,0.000,0.000,1.000


In [9]:
# 7 — archetype probes: mechanics-anchored fixed instrument
from hybrid_search_rrf_dataset.probes import probe

argmax_probe, heads_probe = probe(served), probe(acc)
compare = argmax_probe[["query", "expected"]].copy()
compare["argmax route"] = argmax_probe["served"]
compare["argmax ok"] = argmax_probe["agrees"]
compare["heads route"] = heads_probe["served"]
compare["heads ok"] = heads_probe["agrees"]
for name, frame in (("argmax", argmax_probe), ("heads", heads_probe)):
    decided = frame["agrees"].notna().sum()
    print(f"{name:7s} agrees with mechanics on {frame['agrees'].eq(True).sum()}/{decided}")
compare

argmax  agrees with mechanics on 2/6
heads   agrees with mechanics on 1/6


,query,expected,argmax route,argmax ok,heads route,heads ok
0,a3f5d8b9e12c4d56789abcdef0123456,sparse_only,dense_only,False,dense_only,False
1,/etc/nginx/nginx.conf,sparse_only,sparse_only,True,pure_rrf,False
2,ERR_CONNECTION_RESET,sparse_only,sparse_only,True,sparse_only,True
3,explain quicksort,dense_only,pure_rrf,False,sparse_only,False
4,HTTP 502,NaN,sparse_only,<NA>,dense_only,<NA>
5,comment volent les oiseaux,dense_only,sparse_only,False,sparse_only,False
6,como aprender a programar en rust,dense_only,sparse_only,False,sparse_only,False


# 8 — v2 vs v3, identical config

Two honest readings and one dishonest one. Dishonest: comparing raw held-out
objectives across sources (different test distributions). Honest: (a) each
source's router against ITS OWN constants and oracle (headroom captured);
(b) the transfer matrix — train on one source, evaluate on the other's
held-out decisive rows. Both sources use catalog_v3 features here so the
representation is identical; only the training data changes.

In [10]:
v2_exp = RouterExperiment(labels_path=V2_LABELS, catalog_path=V3_CATALOG,
                          encoder=ENCODER)
side = []
for name, e in (("v2", v2_exp), ("v3", exp)):
    r = e.run(representations=[REPRESENTATION], all_rows=ALL_ROWS,
              max_class_share=MAX_CLASS_SHARE)
    r.insert(0, "source", name)
    side.append(r[["source", *cols]])
pd.concat(side, ignore_index=True).round(3)

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, fitting]/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly f

,source,protocol,representation,all_rows,max_class_share,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured,t_dense,t_sparse
0,v2,random_within_lane,engineered,decisive,None,1021,0.631,0.208,0.367,0.975,0.765,0.391,0.10,0.65
1,v2,holdout_lane,engineered,decisive,None,206,0.607,0.185,0.430,1.000,0.655,0.121,0.30,0.65
2,v3,random_within_lane,engineered,decisive,None,1961,0.504,0.244,0.500,0.992,0.586,0.169,0.55,0.30
3,v3,holdout_lane,engineered,decisive,None,463,0.436,0.268,0.538,1.000,0.490,-0.103,0.55,0.40


In [11]:
# the transfer matrix: rows = trained on, columns = evaluated on (held-out decisive)
tr2, te2 = v2_exp.split(PROTOCOL)
tr3, te3 = exp.split(PROTOCOL)
r2 = StrategyRouter(REPRESENTATION, encoder=ENCODER).fit(tr2, all_rows=ALL_ROWS)
r3 = StrategyRouter(REPRESENTATION, encoder=ENCODER).fit(tr3, all_rows=ALL_ROWS)

def decisive(frame, r):
    return frame[_margin(frame) >= r.decisive_margin]

matrix = {}
for trained, r in (("trained on v2", r2), ("trained on v3", r3)):
    row = {}
    for evaluated, te in (("v2 test", te2), ("v3 test", te3)):
        dec = decisive(te, r)
        row[evaluated] = round(_mean_objective(dec, r.predict_routes(dec)), 3)
    matrix[trained] = row
transfer = pd.DataFrame(matrix).T
for evaluated, te in (("v2 test", te2), ("v3 test", te3)):
    dec = decisive(te, r2)
    transfer.loc["oracle (ceiling)", evaluated] = round(
        _mean_objective(dec, [StrategyName(s) for s in
                              dec[["score_dense_only", "score_pure_rrf", "score_sparse_only"]]
                              .idxmax(axis=1).str.replace("score_", "")]), 3)
transfer

/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return frame.assign(
/Users/andrei/projects/hybrid-search-rrf-dataset/src/hybrid_search_rrf_dataset/router.py:608: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

,v2 test,v3 test
trained on v2,0.751,0.323
trained on v3,0.507,0.571
oracle (ceiling),0.975,0.992


# 9 — the frozen eval reserve

Rows carved out BEFORE any selection, with near-duplicate cluster-mates
excluded from training — the cleanest test set this project owns. The served
router has never seen them in any form.

In [ ]:
import pyarrow.parquet as pq

from composition.pool_v3 import LabelledPool
from composition.recipe import Recipe
from hybrid_search_rrf_dataset.router import _is_engineered

reserve_keys = pd.read_parquet(DATA / "v3" / "eval_reserve.parquet")
pool_frame = LabelledPool(Recipe.v3()).frame()
reserve = pool_frame.merge(
    reserve_keys[["dataset", "query_id"]].astype({"query_id": str}),
    on=["dataset", "query_id"],
)
feat_cols = [c for c in pq.read_schema(V3_CATALOG).names if _is_engineered(c)]
catalog = pd.read_parquet(V3_CATALOG, columns=["dataset", "query_id", *feat_cols])
reserve = reserve.merge(catalog, on=["dataset", "query_id"], how="left")
reserve[feat_cols] = reserve[feat_cols].fillna(0.0)
dec = reserve[_margin(reserve) >= served.decisive_margin]
print(f"reserve: {len(reserve):,} rows, {len(dec):,} decisive")
rows = [("served router", _mean_objective(dec, served.predict_routes(dec)))]
for s in StrategyName:
    rows.append((f"const {s.value}", _mean_objective(dec, [s] * len(dec))))
pd.DataFrame(rows, columns=["policy", "objective (reserve decisive)"]).round(3)